# MAKING OWN LINEAR REGRESSION CLASS FROM SCRATCH :-


In [162]:
import numpy as np
import pandas as pd
class multi_LinearRegression:
    def __init__(self):
        self.B = None

    def fit(self,x_train,y_train):
        
        const_value = 1

        new_col_train = np.full((x_train.shape[0], 1), fill_value=const_value)
       
        X = np.hstack((new_col_train, x_train))
        Y = y_train
        
        self.B = (np.linalg.pinv((X.T)@X))@(X.T)@Y

    def predict(self,x_test):
        
        const_value = 1

        new_col_train = np.full((x_test.shape[0], 1), fill_value=const_value)
       
        x_test = np.hstack((new_col_train, x_test))

        return x_test@(self.B)
    
    def get_parameters(self):
        return self.B

Importing the dataset -- (with 2 features and 1 target variable) :-

In [163]:
df = pd.read_csv("C://Users//Acer//OneDrive//Desktop//Datasets//Real_estate.csv",
                 usecols=['X2 house age','X3 distance to the nearest MRT station','Y house price of unit area'])
df.head()

,X2 house age,X3 distance to the nearest MRT station,Y house price of unit area
0,32.0,84.87882,37.9
1,19.5,306.59470,42.2
2,13.3,561.98450,47.3
3,13.3,561.98450,54.8
4,5.0,390.56840,43.1


In [164]:
df.columns = ['house_age','mrt_dist','price']
df.isnull().sum()

house_age    0
mrt_dist     0
price        0
dtype: int64

In [165]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   house_age  414 non-null    float64
 1   mrt_dist   414 non-null    float64
 2   price      414 non-null    float64
dtypes: float64(3)
memory usage: 9.8 KB


In [166]:
x = df[['house_age','mrt_dist']]
y = df['price']

In [167]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)

In [168]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
x_train = scale.fit_transform(x_train)
x_test = scale.transform(x_test)

Fitting the model with training values.

In [169]:
lr = multi_LinearRegression()
lr.fit(x_train,y_train)

In [170]:
y_pred = lr.predict(x_test)

METRICS :-

In [171]:
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error
r2 = r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = root_mean_squared_error(y_test,y_pred)

print('r2 score : ',r2,'mae : ',mae,'rmse : ',rmse)

r2 score :  0.5755506416768443 mae :  6.247568157853872 rmse :  8.438336086816909


Plotting the regression plane -

In [172]:
import numpy as np
import plotly.graph_objects as go


x1 = x_train[:, 0]
x2 = x_train[:, 1]

y_true = y_train.values
y_pred = lr.predict(x_train)


x1_range = np.linspace(x1.min(), x1.max(), 80)
x2_range = np.linspace(x2.min(), x2.max(), 80)
X1, X2 = np.meshgrid(x1_range, x2_range)

w1 = lr.get_parameters()[1]
w2 = lr.get_parameters()[2]
b = lr.get_parameters()[0]

Z = w1 * X1 + w2 * X2 + b

plane = go.Surface(
    x=X1,
    y=X2,
    z=Z,
    opacity=0.6,
    colorscale='Viridis',
    name='Regression Surface'
)


actual_points = go.Scatter3d(
    x=x1,
    y=x2,
    z=y_true,
    mode='markers',
    marker=dict(size=4, color='red'),
    name='Actual Data'
)

pred_points = go.Scatter3d(
    x=x1,
    y=x2,
    z=y_pred,
    mode='markers',
    marker=dict(size=4, color='blue'),
    name='Predicted Data'
)


residual_lines = []

for i in range(len(x1)):
    residual_lines.append(
        go.Scatter3d(
            x=[x1[i], x1[i]],
            y=[x2[i], x2[i]],
            z=[y_true[i], y_pred[i]],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False
        )
    )


fig = go.Figure()

fig.add_trace(actual_points)
fig.add_trace(pred_points)
fig.add_trace(plane)

for line in residual_lines:
    fig.add_trace(line)


fig.update_layout(
    title=" 3D Linear Regression ",
    scene=dict(
        xaxis_title="House Age",
        yaxis_title="MRT distance",
        zaxis_title="Price per unit"
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

# COMPARING WITH SKLEARN CLASS


In [173]:
from sklearn.linear_model import LinearRegression
lr2 = LinearRegression()
lr2.fit(x_train,y_train)
y_pred = lr2.predict(x_test)
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error
r2 = r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = root_mean_squared_error(y_test,y_pred)

print('r2 score : ',r2,'mae : ',mae,'rmse : ',rmse)


r2 score :  0.5755506416768443 mae :  6.247568157853872 rmse :  8.438336086816909


In [174]:
intercept = lr.get_parameters()[0] 
weights = lr.get_parameters()[1:3]

print(f'self_made class : w = {weights},b = {intercept} ')
print(f'sklearn class : w = {lr2.coef_},b = {lr2.intercept_} ')


self_made class : w = [-2.67408555 -8.93154884],b = 38.39154078549849 
sklearn class : w = [-2.67408555 -8.93154884],b = 38.39154078549849 


# The parameters value for self made class are completely equal to sklearn one . So,--- SUCCESS !!!